# Данный ноутбук проверяет корректность загрузки, а также тестирование модели DenceNet-121.

# Загрузка модели DenseNet-121.

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

# 1. Определяем Dataset
class ChestXrayDataset(Dataset):
    def __init__(self, csv_file, disease_columns, transform=None):
        self.data = pd.read_csv(csv_file)
        self.disease_columns = disease_columns
        self.transform = transform
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]['full_path']
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        labels = self.data.iloc[idx][self.disease_columns].values
        labels = torch.tensor(labels.astype(np.float32))
        
        return image, labels

# 2. Определяем diseases_list
diseases_list = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 
                 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration',
                 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening',
                 'Pneumonia', 'Pneumothorax']

# 3. Трансформации для теста
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 4. Создаем val_loader из тестовой выборки (файл test_data.csv)
print(" Загружаем тестовые данные...")
val_dataset = ChestXrayDataset(
    csv_file='3//test_data.csv',  # файл с тестовой выборкой
    disease_columns=diseases_list,
    transform=val_transform
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f" Тестовых данных: {len(val_dataset)} изображений")

# 5. Загружаем модель если еще не загружена
print("\n Загружаем модель...")
import torchvision.models.densenet
torch.serialization.add_safe_globals([torchvision.models.densenet.DenseNet])

# Укажи правильный путь к файлу модели
model_path = 'full_model_dn_121.pth'  # или 'best_model.pth'
model = torch.load(model_path, weights_only=False)

# 6. Определяем устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Устройство: {device}")

# 7. Перемещаем модель на устройство
model = model.to(device)
model.eval()

print(" Модель готова к тестированию")

# 8. Теперь можно запустить код для тестирования
print("\n" + "=" * 50)
print(" НАЧИНАЕМ ТЕСТИРОВАНИЕ МОДЕЛИ")
print("=" * 50)

📁 Загружаем тестовые данные...
✅ Тестовых данных: 16818 изображений

🧠 Загружаем модель...
📱 Устройство: cuda
✅ Модель готова к тестированию

🧪 НАЧИНАЕМ ТЕСТИРОВАНИЕ МОДЕЛИ


# тестирование модели DenseNet-121.

In [12]:
import torch
import numpy as np
from sklearn.metrics import roc_auc_score

# Определяем устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Устройство: {device}")

# diseases_list должен быть определен
if 'diseases_list' not in locals():
    diseases_list = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 
                     'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration',
                     'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening',
                     'Pneumonia', 'Pneumothorax']

# 1. Убедись, что модель на нужном устройстве и в eval режиме
model = model.to(device)
model.eval()

# 2. Прогнать всю тестовую выборку
all_predictions = []
all_labels = []
all_probabilities = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        probabilities = torch.sigmoid(outputs)
        predictions = (probabilities > 0.4).float() # меняем порог с 0.5 на 0.41
        
        all_predictions.append(predictions.cpu())
        all_labels.append(labels.cpu())
        all_probabilities.append(probabilities.cpu())

# 3. Собрать все результаты
all_predictions = torch.cat(all_predictions).numpy()
all_labels = torch.cat(all_labels).numpy()
all_probabilities = torch.cat(all_probabilities).numpy()

# 4. Посчитать метрики
print(" РЕЗУЛЬТАТЫ НА ТЕСТОВОЙ ВЫБОРКЕ")
print("=" * 50)
print(f"Всего изображений: {len(all_labels)}")

# Для каждого класса считаем метрики
for i, disease in enumerate(diseases_list):
    if np.sum(all_labels[:, i]) > 0:  # Только если класс есть в тестовой выборке
        auc = roc_auc_score(all_labels[:, i], all_probabilities[:, i])
        accuracy = np.mean((all_predictions[:, i] == all_labels[:, i]))
        
        # TP, FP, TN, FN
        tp = np.sum((all_predictions[:, i] == 1) & (all_labels[:, i] == 1))
        fp = np.sum((all_predictions[:, i] == 1) & (all_labels[:, i] == 0))
        tn = np.sum((all_predictions[:, i] == 0) & (all_labels[:, i] == 0))
        fn = np.sum((all_predictions[:, i] == 0) & (all_labels[:, i] == 1))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        print(f"\n{disease}:")
        print(f"  AUC: {auc:.3f} | Accuracy: {accuracy:.3f}")
        print(f"  Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f}")
        print(f"  TP: {tp:3d} | FP: {fp:3d} | TN: {tn:4d} | FN: {fn:3d}")

# 5. Средние метрики по всем классам
print("\n" + "=" * 50)
print(" СРЕДНИЕ МЕТРИКИ:")

micro_auc = roc_auc_score(all_labels, all_probabilities, average='micro')
macro_auc = roc_auc_score(all_labels, all_probabilities, average='macro')
weighted_auc = roc_auc_score(all_labels, all_probabilities, average='weighted')

print(f"Micro AUC: {micro_auc:.3f}")
print(f"Macro AUC: {macro_auc:.3f}")
print(f"Weighted AUC: {weighted_auc:.3f}")

# 6. Пример предсказания для нескольких изображений
print("\n" + "=" * 50)
print("  ПРИМЕРЫ ПРЕДСКАЗАНИЙ:")

with torch.no_grad():
    # Берем первые 3 изображения
    test_images, test_labels = next(iter(val_loader))
    test_images = test_images[:3].to(device)
    test_labels = test_labels[:3]
    
    outputs = model(test_images)
    probabilities = torch.sigmoid(outputs)
    predictions = (probabilities > 0.4).float()
    
    for i in range(3):
        print(f"\nИзображение {i+1}:")
        print(f"  Истинные метки: {[diseases_list[j] for j in range(len(diseases_list)) if test_labels[i][j] == 1]}")
        print(f"  Предсказания: {[diseases_list[j] for j in range(len(diseases_list)) if predictions[i][j] == 1]}")
        
        # Самые уверенные предсказания
        top3_idx = probabilities[i].argsort(descending=True)[:3]
        print(f"  Топ-3 уверенности:")
        for idx in top3_idx:
            print(f"    {diseases_list[idx]}: {probabilities[i][idx]:.3f}")

📱 Устройство: cuda
📊 РЕЗУЛЬТАТЫ НА ТЕСТОВОЙ ВЫБОРКЕ
Всего изображений: 16818

Atelectasis:
  AUC: 0.657 | Accuracy: 0.752
  Precision: 0.179 | Recall: 0.395 | F1: 0.246
  TP: 681 | FP: 3132 | TN: 11962 | FN: 1043

Cardiomegaly:
  AUC: 0.688 | Accuracy: 0.975
  Precision: 0.222 | Recall: 0.005 | F1: 0.010
  TP:   2 | FP:   7 | TN: 16403 | FN: 406

Consolidation:
  AUC: 0.748 | Accuracy: 0.936
  Precision: 0.145 | Recall: 0.116 | F1: 0.129
  TP:  80 | FP: 472 | TN: 15655 | FN: 611

Edema:
  AUC: 0.814 | Accuracy: 0.949
  Precision: 0.104 | Recall: 0.201 | F1: 0.137
  TP:  68 | FP: 585 | TN: 15894 | FN: 271

Effusion:
  AUC: 0.756 | Accuracy: 0.531
  Precision: 0.187 | Recall: 0.880 | F1: 0.309
  TP: 1762 | FP: 7646 | TN: 7170 | FN: 240

Emphysema:
  AUC: 0.756 | Accuracy: 0.978
  Precision: 0.250 | Recall: 0.005 | F1: 0.011
  TP:   2 | FP:   6 | TN: 16446 | FN: 364

Fibrosis:
  AUC: 0.655 | Accuracy: 0.984
  Precision: 0.125 | Recall: 0.004 | F1: 0.007
  TP:   1 | FP:   7 | TN: 16549 | F

# Загрузка модели DenseNet-121.¶

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

# 1. Определяем Dataset
class ChestXrayDataset(Dataset):
    def __init__(self, csv_file, disease_columns, transform=None):
        self.data = pd.read_csv(csv_file)
        self.disease_columns = disease_columns
        self.transform = transform
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]['full_path']
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        labels = self.data.iloc[idx][self.disease_columns].values
        labels = torch.tensor(labels.astype(np.float32))
        
        return image, labels

# 2. Определяем diseases_list
diseases_list = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 
                 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration',
                 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening',
                 'Pneumonia', 'Pneumothorax']

# 3. Трансформации для теста
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 4. Создаем val_loader из тестовой выборки (файл test_data.csv)
print(" Загружаем тестовые данные...")
val_dataset = ChestXrayDataset(
    csv_file='3//test_data.csv',  # файл с тестовой выборкой
    disease_columns=diseases_list,
    transform=val_transform
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f" Тестовых данных: {len(val_dataset)} изображений")

# 5. Загружаем модель если еще не загружена
print("\n Загружаем модель...")
import torchvision.models.densenet
torch.serialization.add_safe_globals([torchvision.models.densenet.DenseNet])

# Укажи правильный путь к файлу модели
model_path = 'full_model_dn_121.pth'  # или 'best_model.pth'
model = torch.load(model_path, weights_only=False)

# 6. Определяем устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Устройство: {device}")

# 7. Перемещаем модель на устройство
model = model.to(device)
model.eval()

print(" Модель готова к тестированию")

# 8. Теперь можно запустить код для тестирования
print("\n" + "=" * 50)
print(" НАЧИНАЕМ ТЕСТИРОВАНИЕ МОДЕЛИ")
print("=" * 50)

# тестирование модели DenseNet-121

In [ ]:
import torch
import numpy as np
from sklearn.metrics import roc_auc_score

# Определяем устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Устройство: {device}")

# diseases_list должен быть определен
if 'diseases_list' not in locals():
    diseases_list = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 
                     'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration',
                     'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening',
                     'Pneumonia', 'Pneumothorax']

#  проверка, что модель на нужном устройстве и в eval режиме
model = model.to(device)
model.eval()

#  Прогнать всю тестовую выборку
all_predictions = []
all_labels = []
all_probabilities = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        probabilities = torch.sigmoid(outputs)
        predictions = (probabilities > 0.4).float() # меняем порог с 0.5 на 0.4
        
        all_predictions.append(predictions.cpu())
        all_labels.append(labels.cpu())
        all_probabilities.append(probabilities.cpu())

# Собрать все результаты
all_predictions = torch.cat(all_predictions).numpy()
all_labels = torch.cat(all_labels).numpy()
all_probabilities = torch.cat(all_probabilities).numpy()

# Посчитать метрики
print(" РЕЗУЛЬТАТЫ НА ТЕСТОВОЙ ВЫБОРКЕ")
print("=" * 50)
print(f"Всего изображений: {len(all_labels)}")

# Для каждого класса считаем метрики
for i, disease in enumerate(diseases_list):
    if np.sum(all_labels[:, i]) > 0:  # Только если класс есть в тестовой выборке
        auc = roc_auc_score(all_labels[:, i], all_probabilities[:, i])
        accuracy = np.mean((all_predictions[:, i] == all_labels[:, i]))
        
        # TP, FP, TN, FN
        tp = np.sum((all_predictions[:, i] == 1) & (all_labels[:, i] == 1))
        fp = np.sum((all_predictions[:, i] == 1) & (all_labels[:, i] == 0))
        tn = np.sum((all_predictions[:, i] == 0) & (all_labels[:, i] == 0))
        fn = np.sum((all_predictions[:, i] == 0) & (all_labels[:, i] == 1))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        print(f"\n{disease}:")
        print(f"  AUC: {auc:.3f} | Accuracy: {accuracy:.3f}")
        print(f"  Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f}")
        print(f"  TP: {tp:3d} | FP: {fp:3d} | TN: {tn:4d} | FN: {fn:3d}")

# Средние метрики по всем классам
print("\n" + "=" * 50)
print(" СРЕДНИЕ МЕТРИКИ:")

micro_auc = roc_auc_score(all_labels, all_probabilities, average='micro')
macro_auc = roc_auc_score(all_labels, all_probabilities, average='macro')
weighted_auc = roc_auc_score(all_labels, all_probabilities, average='weighted')

print(f"Micro AUC: {micro_auc:.3f}")
print(f"Macro AUC: {macro_auc:.3f}")
print(f"Weighted AUC: {weighted_auc:.3f}")

#  Пример предсказания для нескольких изображений
print("\n" + "=" * 50)
print("  ПРИМЕРЫ ПРЕДСКАЗАНИЙ:")

with torch.no_grad():
    # Берем первые 3 изображения
    test_images, test_labels = next(iter(val_loader))
    test_images = test_images[:3].to(device)
    test_labels = test_labels[:3]
    
    outputs = model(test_images)
    probabilities = torch.sigmoid(outputs)
    predictions = (probabilities > 0.4).float()
    
    for i in range(3):
        print(f"\nИзображение {i+1}:")
        print(f"  Истинные метки: {[diseases_list[j] for j in range(len(diseases_list)) if test_labels[i][j] == 1]}")
        print(f"  Предсказания: {[diseases_list[j] for j in range(len(diseases_list)) if predictions[i][j] == 1]}")
        
        # Самые уверенные предсказания
        top3_idx = probabilities[i].argsort(descending=True)[:3]
        print(f"  Топ-3 уверенности:")
        for idx in top3_idx:
            print(f"    {diseases_list[idx]}: {probabilities[i][idx]:.3f}")